# Preprocessing and mapping exploration

Explores offline preprocessing into `mapped_preprocessed_datasets/`: joint class ids (0 = ignore) stored as **uint16** dense label PNGs when there are more than 255 classes, taxonomy + trinary concept LUTs from [`class_to_concepts.csv`](../../configs/class_to_concepts.csv), long-edge resize (max 2048 px), and benchmarks for materialising multi-hot concept tensors.

In [ ]:
%load_ext autoreload
%autoreload 2

## 1. Config

In [ ]:
from __future__ import annotations

import io
import json
import shutil
import time
from pathlib import Path

import boto3
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from matplotlib.patches import Patch
from matplotlib.colors import ListedColormap
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from tqdm import tqdm

from mermaidseg.datasets.dataset import (
    CoralNetDataset,
    CoralscapesDataset,
    ExternalDenseCoralDataset,
    ExternalSparseCoralDataset,
    MermaidDataset,
)
from mermaidseg.datasets.labelspace import (
    TAXONOMY_COLUMNS,
    Labelspace,
    binary_concept_lut_array,
    build_binary_concepts,
    build_class_lut,
    build_labelspace,
    build_taxonomy_levels,
    load_class_to_concepts_csv,
    taxonomy_lut_arrays,
    write_labelspace_artifacts,
)
from mermaidseg.io import _load_csv_if_path

In [ ]:
# Toggle joint label space definition: "csv_canonical" | "mermaid_api"
LABELSPACE_MODE = "csv_canonical"

REPO_ROOT = Path("../..").resolve()
CLASS_CSV = REPO_ROOT / "configs" / "class_to_concepts.csv"
# Written under repo root; listed in .gitignore
OUTPUT_ROOT = REPO_ROOT / "scratch_mapped_preprocessed_datasets"
UPLOAD_TO_S3 = False  # set True + configure bucket/key prefix to push OUTPUT_ROOT
S3_BUCKET = "dev-datamermaid-sm-sources"
S3_PREFIX = "mapped_preprocessed_datasets"

MAX_LONG_EDGE = 2048
CLASS_LABEL_DTYPE = np.uint16  # dense joint-class PNGs (supports >255 classes)
N_SAMPLES_PER_DATASET = 3
BENCHMARK_BATCH_SIZE = 4
BENCHMARK_NUM_WORKERS = (0, 2)
BENCHMARK_BATCHES = 32

WHITELIST_SOURCES = _load_csv_if_path(REPO_ROOT / "configs" / "run1_train_sources.csv")

SOURCE_BUCKET = "dev-datamermaid-sm-sources"
OPT_PREFIX = "optimized_datasets/"

## 2. Build labelspace, taxonomy, binary concepts, class LUT

Run **both** modes below to compare class counts; training uses `LABELSPACE_MODE` only.

In [ ]:
def build_full_labelspace_bundle(mode: str):
    ls = build_labelspace(mode, CLASS_CSV)
    df_csv = load_class_to_concepts_csv(CLASS_CSV)
    tax = build_taxonomy_levels(df_csv)
    bspec = build_binary_concepts(df_csv)
    lut = build_class_lut(ls, tax, bspec)
    return ls, tax, bspec, lut


bundle_csv = build_full_labelspace_bundle("csv_canonical")
bundle_api = build_full_labelspace_bundle("mermaid_api")

compare = pd.DataFrame(
    {
        "mode": ["csv_canonical", "mermaid_api"],
        "n_classes": [len(bundle_csv[0].id2name), len(bundle_api[0].id2name)],
        "n_binary_concepts": [len(bundle_csv[2].columns), len(bundle_api[2].columns)],
    }
)
for col in TAXONOMY_COLUMNS:
    compare[f"cardinality_{col}"] = [
        len(bundle_csv[1][col].id2value) - 1,
        len(bundle_api[1][col].id2value) - 1,
    ]
compare

In [ ]:
labelspace, taxonomy, binary_spec, class_lut = build_full_labelspace_bundle(LABELSPACE_MODE)

tax_lut = taxonomy_lut_arrays(class_lut)
bin_lut = binary_concept_lut_array(class_lut, binary_spec.columns)

LABELSPACE_DIR = OUTPUT_ROOT / "_labelspace"
write_labelspace_artifacts(LABELSPACE_DIR, labelspace, taxonomy, binary_spec, class_lut)
print("Wrote", LABELSPACE_DIR)
print("class_lut shape", class_lut.shape, "tax_lut", tax_lut.shape, "bin_lut", bin_lut.shape)

## 3. Resize + preprocess helpers

In [ ]:
def resize_plan(h: int, w: int, max_long: int = MAX_LONG_EDGE):
    long = max(h, w)
    scale = min(1.0, max_long / float(long)) if long > 0 else 1.0
    nh, nw = max(1, int(round(h * scale))), max(1, int(round(w * scale)))
    return scale, nh, nw


def resize_rgb(np_hwc: np.ndarray, nh: int, nw: int) -> np.ndarray:
    im = Image.fromarray(np_hwc)
    im = im.resize((nw, nh), Image.Resampling.LANCZOS)
    return np.asarray(im)


def resize_mask_nearest(mask: np.ndarray, nh: int, nw: int) -> np.ndarray:
    arr = np.asarray(mask, dtype=CLASS_LABEL_DTYPE)
    im = Image.fromarray(arr)
    im = im.resize((nw, nh), Image.Resampling.NEAREST)
    return np.asarray(im, dtype=CLASS_LABEL_DTYPE)


def filter_sparse_annotations(
    ann: pd.DataFrame,
    shape_hw: tuple[int, int],
    labelspace: Labelspace,
    csv_source_key: str,
) -> pd.DataFrame:
    h, w = shape_hw
    rows = []
    for _, r in ann.iterrows():
        if pd.isna(r.get("benthic_attribute_name")):
            continue
        name = str(r["benthic_attribute_name"]).strip()
        if name == "":
            continue
        ri, ci = int(r["row"]), int(r["col"])
        if ri < 0 or ci < 0 or ri >= h or ci >= w:
            continue
        cid = labelspace.source_name_to_class_id(csv_source_key, name)
        if cid == 0:
            continue
        rows.append({"row": ri, "col": ci, "class_id": int(cid)})
    return pd.DataFrame(rows)


def scale_points(df: pd.DataFrame, scale: float) -> pd.DataFrame:
    if df.empty:
        return df
    out = df.copy()
    out["row"] = (out["row"] * scale).astype(int)
    out["col"] = (out["col"] * scale).astype(int)
    return out


# Coralscapes local id → CSV source name (lowercase keys in class_to_concepts)
ID2LABEL_CORALSCAPES = {
    "1": "seagrass",
    "2": "trash",
    "3": "other coral dead",
    "4": "other coral bleached",
    "5": "sand",
    "6": "other coral alive",
    "7": "human",
    "8": "transect tools",
    "9": "fish",
    "10": "algae covered substrate",
    "11": "other animal",
    "12": "unknown hard substrate",
    "13": "background",
    "14": "dark",
    "15": "transect line",
    "16": "massive/meandering bleached",
    "17": "massive/meandering alive",
    "18": "rubble",
    "19": "branching bleached",
    "20": "branching dead",
    "21": "millepora",
    "22": "branching alive",
    "23": "massive/meandering dead",
    "24": "clam",
    "25": "acropora alive",
    "26": "sea cucumber",
    "27": "turbinaria",
    "28": "table acropora alive",
    "29": "sponge",
    "30": "anemone",
    "31": "pocillopora alive",
    "32": "table acropora dead",
    "33": "meandering bleached",
    "34": "stylophora alive",
    "35": "sea urchin",
    "36": "meandering alive",
    "37": "meandering dead",
    "38": "crown of thorn",
    "39": "dead clam",
}


def coralscapes_mask_to_joint(mask: np.ndarray, ls: Labelspace) -> np.ndarray:
    out = np.zeros(mask.shape, dtype=CLASS_LABEL_DTYPE)
    for old in np.unique(mask):
        o = int(old)
        if o == 0:
            continue
        name = ID2LABEL_CORALSCAPES.get(str(o))
        if name is None:
            continue
        cid = ls.source_name_to_class_id("coralscapes", name)
        out[mask == old] = cid
    return out


def external_dense_mask_to_joint(mask: np.ndarray, ds: ExternalDenseCoralDataset, ls: Labelspace, csv_key: str) -> np.ndarray:
    out = np.zeros(mask.shape, dtype=CLASS_LABEL_DTYPE)
    for v in np.unique(mask):
        vi = int(v)
        if vi == 0:
            continue
        name = ds.id2label.get(vi)
        if name is None:
            continue
        cid = ls.source_name_to_class_id(csv_key, str(name))
        out[mask == v] = cid
    return out


def append_manifest(path: Path, row: dict):
    path.parent.mkdir(parents=True, exist_ok=True)
    rows = []
    if path.exists():
        rows = pd.read_parquet(path).to_dict("records")
    rows.append(row)
    pd.DataFrame(rows).to_parquet(path, index=False)

## 4. Preprocess a few samples per dataset

In [ ]:
if OUTPUT_ROOT.exists():
    shutil.rmtree(OUTPUT_ROOT)
OUTPUT_ROOT.mkdir(parents=True)
write_labelspace_artifacts(OUTPUT_ROOT / "_labelspace", labelspace, taxonomy, binary_spec, class_lut)

stats_rows = []


def write_dense_sample(
    ds_name: str,
    csv_key: str,
    image_u8: np.ndarray,
    joint_mask: np.ndarray,
    source_id: str,
    image_id: str,
):
    h, w = image_u8.shape[:2]
    scale, nh, nw = resize_plan(h, w)
    img_r = resize_rgb(image_u8, nh, nw)
    m_r = resize_mask_nearest(joint_mask, nh, nw)
    rel_img = Path("images") / str(source_id) / f"{image_id}.jpg"
    rel_lbl = Path("labels") / str(source_id) / f"{image_id}.png"
    out_dir = OUTPUT_ROOT / ds_name
    (out_dir / rel_img).parent.mkdir(parents=True, exist_ok=True)
    (out_dir / rel_lbl).parent.mkdir(parents=True, exist_ok=True)
    Image.fromarray(img_r).save(out_dir / rel_img, quality=95)
    Image.fromarray(m_r).save(out_dir / rel_lbl)
    append_manifest(
        out_dir / "sample_manifest.parquet",
        {
            "source_id": str(source_id),
            "image_id": str(image_id),
            "image_key": str(rel_img.as_posix()),
            "mask_key": str(rel_lbl.as_posix()),
            "annotations_key": None,
            "label_type": "dense",
            "orig_h": h,
            "orig_w": w,
            "new_h": nh,
            "new_w": nw,
            "scale": float(scale),
        },
    )
    stats_rows.append({"dataset": ds_name, "resized": scale < 1.0, "kind": "dense"})


def write_sparse_sample(
    ds_name: str,
    csv_key: str,
    image_u8: np.ndarray,
    ann_filtered: pd.DataFrame,
    source_id: str,
    image_id: str,
):
    h, w = image_u8.shape[:2]
    scale, nh, nw = resize_plan(h, w)
    img_r = resize_rgb(image_u8, nh, nw)
    ann_s = scale_points(ann_filtered, scale)
    ann_s = ann_s[(ann_s["row"] >= 0) & (ann_s["col"] >= 0) & (ann_s["row"] < nh) & (ann_s["col"] < nw)]
    rel_img = Path("images") / str(source_id) / f"{image_id}.jpg"
    rel_ann = Path("annotations") / str(source_id) / f"{image_id}.parquet"
    out_dir = OUTPUT_ROOT / ds_name
    (out_dir / rel_img).parent.mkdir(parents=True, exist_ok=True)
    (out_dir / rel_ann).parent.mkdir(parents=True, exist_ok=True)
    Image.fromarray(img_r).save(out_dir / rel_img, quality=95)
    ann_s.to_parquet(out_dir / rel_ann, index=False)
    append_manifest(
        out_dir / "sample_manifest.parquet",
        {
            "source_id": str(source_id),
            "image_id": str(image_id),
            "image_key": str(rel_img.as_posix()),
            "mask_key": None,
            "annotations_key": str(rel_ann.as_posix()),
            "label_type": "sparse",
            "orig_h": h,
            "orig_w": w,
            "new_h": nh,
            "new_w": nw,
            "scale": float(scale),
        },
    )
    stats_rows.append({"dataset": ds_name, "resized": scale < 1.0, "kind": "sparse", "n_ann": len(ann_s)})


rng = np.random.default_rng(42)

In [ ]:
# MERMAID (sparse)
md = MermaidDataset(transform=None, padding=None, concept_mapping_flag=False)
idxs = rng.choice(len(md), size=min(N_SAMPLES_PER_DATASET, len(md)), replace=False)
for idx in idxs:
    image_id = md.df_images.loc[int(idx), "image_id"]
    row_kwargs = md.df_images.loc[int(idx)].to_dict()
    src = str(row_kwargs.get("region_id", "mermaid"))
    img = md.read_image(**row_kwargs)
    ann = md.df_annotations.loc[md.df_annotations["image_id"] == image_id, ["row", "col", "benthic_attribute_name"]]
    ann_f = filter_sparse_annotations(ann, img.shape[:2], labelspace, "mermaid")
    write_sparse_sample("mermaid", "mermaid", img, ann_f, src, str(image_id))
print("mermaid done")

In [ ]:
# CoralNet (sparse)
cn = CoralNetDataset(
    transform=None,
    padding=None,
    concept_mapping_flag=False,
    whitelist_sources=WHITELIST_SOURCES,
)
idxs = rng.choice(len(cn), size=min(N_SAMPLES_PER_DATASET, len(cn)), replace=False)
for idx in idxs:
    image_id = cn.df_images.loc[int(idx), "image_id"]
    source_id = str(cn.df_images.loc[int(idx), "source_id"])
    row_kwargs = cn.df_images.loc[int(idx)].to_dict()
    img = cn.read_image(**row_kwargs)
    ann = cn.df_annotations.loc[
        (cn.df_annotations["image_id"] == image_id) & (cn.df_annotations["source_id"].astype(str) == source_id),
        ["row", "col", "benthic_attribute_name"],
    ]
    ann_f = filter_sparse_annotations(ann, img.shape[:2], labelspace, "coralnet")
    write_sparse_sample("coralnet", "coralnet", img, ann_f, source_id, str(image_id))
print("coralnet done")

In [ ]:
# Coralscapes (dense) — raw HF split
cs = CoralscapesDataset(split="train", transform=None, concept_mapping_flag=False)
idxs = rng.choice(len(cs), size=min(N_SAMPLES_PER_DATASET, len(cs)), replace=False)
for idx in idxs:
    ex = cs.dataset[int(idx)]
    img = np.asarray(ex["image"])
    mask = np.asarray(ex["label"])
    joint = coralscapes_mask_to_joint(mask, labelspace)
    image_id = f"coralscapes_train_{int(idx)}"
    write_dense_sample("coralscapes", "coralscapes", img, joint, "coralscapes", image_id)
print("coralscapes done")

In [ ]:
def preprocess_external_dense(name: str, csv_key: str):
    ds = ExternalDenseCoralDataset(
        source_bucket=SOURCE_BUCKET,
        source_prefix=OPT_PREFIX,
        dataset_name=name,
        transform=None,
    )
    idxs = rng.choice(len(ds), size=min(N_SAMPLES_PER_DATASET, len(ds)), replace=False)
    for idx in idxs:
        sample = ds.samples[int(idx)]
        image_id = sample["image_id"]
        source_id = str(sample["source_id"])
        image_key = ds.image_key_by_id[image_id]
        mask_key = ds.mask_key_by_id[image_id]
        s3 = boto3.client("s3")
        img_b = s3.get_object(Bucket=SOURCE_BUCKET, Key=image_key)["Body"].read()
        m_b = s3.get_object(Bucket=SOURCE_BUCKET, Key=mask_key)["Body"].read()
        img = np.asarray(Image.open(io.BytesIO(img_b)).convert("RGB"))
        m = np.asarray(Image.open(io.BytesIO(m_b)))
        if m.ndim == 3:
            m = m[..., 0]
        joint = external_dense_mask_to_joint(np.asarray(m), ds, labelspace, csv_key)
        write_dense_sample(name, csv_key, img, joint, source_id, str(image_id))


preprocess_external_dense("mosaics_ucsd", "mosaics_ucsd")
preprocess_external_dense("benthos_yuval", "benthos_yuval")
print("external dense done")

In [ ]:
def preprocess_external_sparse(name: str, csv_key: str):
    ds = ExternalSparseCoralDataset(
        source_bucket=SOURCE_BUCKET,
        source_prefix=OPT_PREFIX,
        dataset_name=name,
        transform=None,
        padding=None,
        concept_mapping_flag=False,
    )
    idxs = rng.choice(len(ds), size=min(N_SAMPLES_PER_DATASET, len(ds)), replace=False)
    for idx in idxs:
        image_id = ds.df_images.loc[int(idx), "image_id"]
        source_id = str(ds.df_images.loc[int(idx), "source_id"])
        row_kwargs = ds.df_images.loc[int(idx)].to_dict()
        img = ds.read_image(image_id, **row_kwargs)
        ann = ds.df_annotations.loc[
            (ds.df_annotations["image_id"] == image_id) & (ds.df_annotations["source_id"].astype(str) == source_id),
            ["row", "col", "benthic_attribute_name"],
        ]
        ann_f = filter_sparse_annotations(ann, img.shape[:2], labelspace, csv_key)
        write_sparse_sample(name, csv_key, img, ann_f, source_id, str(image_id))


for nm, key in [
    ("catlin_seaview", "catlin_seaview"),
    ("moorea_labeled_corals", "moorea_labeled_corals"),
    ("pacific_labeled_corals", "pacific_labeled_corals"),
]:
    preprocess_external_sparse(nm, key)
print("external sparse done")

pd.DataFrame(stats_rows)

## 5. Visualisation (side-by-side)

Joint class mask (dense) or point overlay (sparse); taxonomy channels from class LUT; binary-concept fingerprint from one annotated pixel.

In [ ]:
def legend_patches(id2name: dict[int, str], present_ids: list[int], max_items: int = 24):
    present_ids = sorted(set(present_ids))[:max_items]
    cmap = plt.get_cmap("tab20")
    handles = []
    for j, pid in enumerate(present_ids):
        name = id2name.get(pid, str(pid))
        handles.append(Patch(facecolor=cmap(j % 20), edgecolor="k", label=f"{pid}: {name}"))
    return handles


def taxonomy_masks_from_class_mask(m: np.ndarray, tax_lut_arr: np.ndarray) -> dict[str, np.ndarray]:
    """Per-pixel taxonomy ids (0 = ignore) from dense class mask."""
    out = {}
    flat = m.astype(np.int64).ravel()
    for i, col in enumerate(TAXONOMY_COLUMNS):
        mapped = np.zeros_like(flat)
        ok = flat > 0
        mapped[ok] = tax_lut_arr[flat[ok], i]
        out[col] = mapped.reshape(m.shape)
    return out


def fingerprint_bar(ax, class_id: int, class_lut_df: pd.DataFrame, binary_cols: list[str]):
    row = class_lut_df.loc[class_id]
    vals = [int(row[f"bin_{c}"]) for c in binary_cols]
    colors = ["#cccccc" if v == 0 else "#c44e52" if v == 1 else "#55a868" for v in vals]
    ax.barh(range(len(vals)), [1] * len(vals), color=colors, height=0.8)
    ax.set_yticks(range(len(binary_cols)))
    ax.set_yticklabels(binary_cols, fontsize=6)
    ax.set_xticks([])
    ax.set_title(f"Binary concepts (id={class_id})")


def show_dataset_grid(ds_name: str, max_plots: int = 3):
    man = pd.read_parquet(OUTPUT_ROOT / ds_name / "sample_manifest.parquet")
    man = man.head(max_plots)
    base = OUTPUT_ROOT / ds_name
    id2n = labelspace.id2name

    for _, r in man.iterrows():
        img_path = base / r["image_key"]
        im = np.asarray(Image.open(img_path).convert("RGB"))
        fig, axes = plt.subplots(2, 4, figsize=(18, 9))
        fig.suptitle(f"{ds_name}  {r['image_id']}  ({r['label_type']})")
        axes[0, 0].imshow(im)
        axes[0, 0].set_title("Image")
        axes[0, 0].axis("off")

        sample_class = None
        ann: pd.DataFrame | None = None
        present_ids: list[int] = []
        if r["label_type"] == "dense":
            m = np.asarray(Image.open(base / r["mask_key"]))
            present = [int(x) for x in np.unique(m) if x > 0]
            present_ids = present
            axes[0, 1].imshow(m, cmap="tab20", vmin=0, vmax=max(1, max(present, default=1)))
            axes[0, 1].set_title("Joint class id")
            axes[0, 1].axis("off")
            tax = taxonomy_masks_from_class_mask(m, tax_lut)
            sample_class = int(m[m > 0].min()) if (m > 0).any() else None
        else:
            ann = pd.read_parquet(base / r["annotations_key"])
            axes[0, 1].imshow(im)
            if not ann.empty:
                axes[0, 1].scatter(ann["col"], ann["row"], c=ann["class_id"], cmap="tab20", s=12)
                sample_class = int(ann.iloc[0]["class_id"])
                present_ids = [int(x) for x in ann["class_id"].unique()]
            axes[0, 1].set_title("Sparse points (colored by class_id)")
            axes[0, 1].axis("off")
            # pseudo mask for taxonomy row: empty
            tax = {c: np.zeros(im.shape[:2], dtype=np.int32) for c in TAXONOMY_COLUMNS}
            if not ann.empty:
                for _, a in ann.iterrows():
                    cid = int(a["class_id"])
                    for j, col in enumerate(TAXONOMY_COLUMNS):
                        tax[col][int(a["row"]), int(a["col"])] = tax_lut[cid, j]

        # Taxonomy strip (first 3 levels)
        for k, col in enumerate(TAXONOMY_COLUMNS[:3]):
            axes[1, k].imshow(tax[col], cmap="tab10", vmin=0, vmax=max(tax[col].max(), 1))
            spec = taxonomy[col]
            present_t = [int(x) for x in np.unique(tax[col]) if x > 0]
            leg = [f"{p}:{spec.id2value.get(p, p)}" for p in present_t[:8]]
            axes[1, k].set_title(col + "\n" + ", ".join(leg))
            axes[1, k].axis("off")

        if sample_class is not None:
            fingerprint_bar(axes[1, 3], sample_class, class_lut, binary_spec.columns)
        else:
            axes[1, 3].axis("off")

        # Legends for joint classes
        leg_handles = legend_patches(id2n, present_ids) if present_ids else []
        if leg_handles:
            axes[0, 2].legend(handles=leg_handles, loc="center", fontsize=7, title="Joint classes")
        axes[0, 2].axis("off")

        axes[0, 3].axis("off")
        plt.tight_layout()
        plt.show()


for ds in [
    "mermaid",
    "coralnet",
    "coralscapes",
    "mosaics_ucsd",
    "benthos_yuval",
    "catlin_seaview",
    "moorea_labeled_corals",
    "pacific_labeled_corals",
]:
    p = OUTPUT_ROOT / ds / "sample_manifest.parquet"
    if p.exists():
        show_dataset_grid(ds, max_plots=2)

## 6. Concept storage benchmark (A–D)

- **A** — class PNG + runtime `bin_lut[class_id]` gather (no extra concept files).
- **B** — 33 separate uint8 PNGs per sample (`0/1/2`).
- **C** — single `.npz` with `(H,W,33)` uint8.
- **D** — 11 RGB PNGs packing 3 concepts per file.

Expect **A** to win on disk + decode time; B/C/D help when the class→concept map is not a pure LUT.

In [ ]:
def materialise_concepts_B(base: Path, image_id: str, source_id: str, joint_mask: np.ndarray):
    d = base / "concepts_B" / str(source_id)
    d.mkdir(parents=True, exist_ok=True)
    flat = joint_mask.astype(np.int64).ravel()
    chans = bin_lut[flat].reshape(*joint_mask.shape, bin_lut.shape[1]).astype(np.uint8)
    for k in range(chans.shape[2]):
        Image.fromarray(chans[..., k]).save(d / f"{image_id}_c{k:02d}.png")


def materialise_concepts_C(base: Path, image_id: str, source_id: str, joint_mask: np.ndarray):
    d = base / "concepts_C" / str(source_id)
    d.mkdir(parents=True, exist_ok=True)
    flat = joint_mask.astype(np.int64).ravel()
    chans = bin_lut[flat].reshape(*joint_mask.shape, bin_lut.shape[1]).astype(np.uint8)
    np.savez_compressed(d / f"{image_id}.npz", concepts=chans)


def materialise_concepts_D(base: Path, image_id: str, source_id: str, joint_mask: np.ndarray):
    d = base / "concepts_D" / str(source_id)
    d.mkdir(parents=True, exist_ok=True)
    flat = joint_mask.astype(np.int64).ravel()
    chans = bin_lut[flat].reshape(*joint_mask.shape, bin_lut.shape[1]).astype(np.uint8)
    n = chans.shape[2]
    for pack in range(0, n, 3):
        slab = chans[..., pack : pack + 3]
        pad = 3 - slab.shape[2]
        if pad:
            slab = np.concatenate([slab, np.zeros((*slab.shape[:2], pad), dtype=np.uint8)], axis=2)
        Image.fromarray(slab, mode="RGB").save(d / f"{image_id}_pack{pack // 3:02d}.png")


def dir_size_bytes(p: Path) -> int:
    total = 0
    for f in p.rglob("*"):
        if f.is_file():
            total += f.stat().st_size
    return total


def count_files(p: Path) -> int:
    return sum(1 for f in p.rglob("*") if f.is_file())


# Build variant files from first coralscapes sample (dense)
bench_ds = "coralscapes"
man = pd.read_parquet(OUTPUT_ROOT / bench_ds / "sample_manifest.parquet").iloc[:2]
base = OUTPUT_ROOT / bench_ds
for _, r in man.iterrows():
    m = np.asarray(Image.open(base / r["mask_key"]))
    materialise_concepts_B(base, str(r["image_id"]), str(r["source_id"]), m)
    materialise_concepts_C(base, str(r["image_id"]), str(r["source_id"]), m)
    materialise_concepts_D(base, str(r["image_id"]), str(r["source_id"]), m)

variant_roots = {
    "B": base / "concepts_B",
    "C": base / "concepts_C",
    "D": base / "concepts_D",
}
rows = []
for k, root in variant_roots.items():
    rows.append({"variant": k, "bytes": dir_size_bytes(root), "n_files": count_files(root)})
rows.append({"variant": "A (no extra files)", "bytes": 0, "n_files": 0})
pd.DataFrame(rows)

In [ ]:
class BenchDataset(Dataset):
    def __init__(self, mode: str, manifest: pd.DataFrame, base: Path):
        self.mode = mode
        self.manifest = manifest.reset_index(drop=True)
        self.base = base

    def __len__(self):
        return len(self.manifest)

    def __getitem__(self, idx):
        r = self.manifest.iloc[idx]
        m = np.asarray(Image.open(self.base / r["mask_key"])).astype(np.int64)
        if self.mode == "A":
            flat = m.ravel()
            concepts = bin_lut[flat].reshape(*m.shape, bin_lut.shape[1])
        elif self.mode == "B":
            sid, iid = str(r["source_id"]), str(r["image_id"])
            chans = []
            for k in range(bin_lut.shape[1]):
                ch = np.asarray(Image.open(self.base / "concepts_B" / sid / f"{iid}_c{k:02d}.png"))
                chans.append(ch)
            concepts = np.stack(chans, axis=-1)
        elif self.mode == "C":
            z = np.load(self.base / "concepts_C" / str(r["source_id"]) / f"{r['image_id']}.npz")
            concepts = z["concepts"]
        elif self.mode == "D":
            sid, iid = str(r["source_id"]), str(r["image_id"])
            packs = sorted((self.base / "concepts_D" / sid).glob(f"{iid}_pack*.png"))
            chans = []
            for p in packs:
                rgb = np.asarray(Image.open(p).convert("RGB"))
                for c in range(3):
                    chans.append(rgb[..., c])
            concepts = np.stack(chans[: bin_lut.shape[1]], axis=-1)
        else:
            raise ValueError(self.mode)
        return m.astype(np.float32), concepts.astype(np.float32)


def run_benchmark(mode: str, num_workers: int):
    ds = BenchDataset(mode, man, base)
    loader = DataLoader(
        ds,
        batch_size=BENCHMARK_BATCH_SIZE,
        num_workers=num_workers,
        shuffle=False,
    )
    t0 = time.perf_counter()
    n_batches = 0
    for batch in loader:
        n_batches += 1
        if n_batches >= BENCHMARK_BATCHES:
            break
    return (time.perf_counter() - t0) / n_batches


bench_results = []
for mode in ["A", "B", "C", "D"]:
    for nw in BENCHMARK_NUM_WORKERS:
        tpb = run_benchmark(mode, nw)
        bench_results.append({"variant": mode, "num_workers": nw, "sec_per_batch": tpb})
pd.DataFrame(bench_results)

In [ ]:
# Optional: GPU gather cost for variant A (microbenchmark)
if torch.cuda.is_available():
    m = torch.randint(0, bin_lut.shape[0], (512, 512), device="cuda")
    lut_t = torch.from_numpy(bin_lut).to("cuda")
    # warmup
    for _ in range(10):
        _ = lut_t[m.long()]
    torch.cuda.synchronize()
    t0 = time.perf_counter()
    for _ in range(100):
        _ = lut_t[m.long()]
    torch.cuda.synchronize()
    print("GPU LUT gather (100x 512²):", time.perf_counter() - t0, "s")
else:
    print("CUDA not available — skipped GPU microbenchmark")

## 7. Optional S3 upload

Requires AWS credentials (e.g. profile `mermaid-core`).

In [ ]:
if UPLOAD_TO_S3:
    s3 = boto3.client("s3")
    for path in OUTPUT_ROOT.rglob("*"):
        if path.is_file():
            rel = path.relative_to(OUTPUT_ROOT)
            key = f"{S3_PREFIX}/{rel.as_posix()}"
            s3.upload_file(str(path), S3_BUCKET, key)
            print("uploaded", key)
else:
    print("UPLOAD_TO_S3=False — skipped")